In [1]:
# ============================================
# Johansen Cointegration Test Script
# - Input  : ./merged_var_input.csv
# - Output : ./johansen_result.csv
# ============================================

import pandas as pd
import numpy as np
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# =============================
# 1. Load data
# =============================
DATA_PATH = "./merged_var_input.csv"
OUTPUT_PATH = "./johansen_result.csv"

df = pd.read_csv(DATA_PATH)

# Date 컬럼 제거 (있으면)
if "Date" in df.columns:
    df = df.drop(columns=["Date"])

# diff 변수 자동 제외
level_cols = [
    col for col in df.columns
    if not col.startswith("dlog_")
    and not col.startswith("d_")
]

data_level = df[level_cols].dropna()

print("사용 level 변수:")
print(level_cols)
print("표본 수:", len(data_level))
print("-" * 50)

# =============================
# 2. Johansen Test
# =============================
# det_order:
#   -1: no deterministic term
#    0: constant
#    1: linear trend
det_order = 0

# k_ar_diff = VAR lag - 1
# lag=1이면 0 또는 1 시도 가능
k_ar_diff = 1

jres = coint_johansen(data_level, det_order, k_ar_diff)

# =============================
# 3. 결과 정리
# =============================
results = []

for i in range(len(level_cols)):
    results.append({
        "r": i,
        "trace_stat": jres.lr1[i],
        "trace_95crit": jres.cvt[i, 1],
        "trace_reject_95": jres.lr1[i] > jres.cvt[i, 1],
        "maxeig_stat": jres.lr2[i],
        "maxeig_95crit": jres.cvm[i, 1],
        "maxeig_reject_95": jres.lr2[i] > jres.cvm[i, 1],
        "eigenvalue": jres.eig[i]
    })

result_df = pd.DataFrame(results)

# =============================
# 4. 공적분 rank 계산
# =============================
# Trace 기준
rank_trace = sum(result_df["trace_reject_95"])

# MaxEig 기준
rank_maxeig = sum(result_df["maxeig_reject_95"])

print("Trace 기준 공적분 rank:", rank_trace)
print("MaxEigen 기준 공적분 rank:", rank_maxeig)
print("-" * 50)

# =============================
# 5. CSV 저장
# =============================
result_df.to_csv(OUTPUT_PATH, index=False)

print("결과 저장 완료:", OUTPUT_PATH)

사용 level 변수:
['SOLVPN', 'COPPER', 'DXY', 'UST10Y', 'VIX']
표본 수: 1325
--------------------------------------------------
Trace 기준 공적분 rank: 1
MaxEigen 기준 공적분 rank: 1
--------------------------------------------------
결과 저장 완료: ./johansen_result.csv
